# ProteinMPNN peptide-probability specificity test

Does each designed binder add information that discriminates the KRAS G12D
aspartate from the wild-type glycine?

ProteinMPNN is an inverse-folding model: given a backbone and a partial
sequence, it returns a probability distribution over amino acids at each
masked position. Mask the 10-mer peptide in a designed complex and the model
reports which residue the *structure* wants at each peptide position.

**The control is the whole experiment.** The HLA-A\*11:01 groove already
prefers aspartate at p6 on its own - that is why this peptide binds. So a
high P(Asp) in the complex proves nothing about the binder. Every design is
therefore run twice: once as the full complex, once with the binder chain
deleted. The reported quantity is the shift between them.

    delta = log[P(D)/P(G)]_complex  -  log[P(D)/P(G)]_apo

    delta > 0   the binder adds discriminating information
    delta = 0   the binder is transparent to the mutation
    delta < 0   the binder occludes the groove's own readout

This is the specificity filter used in the Baker lab's 2025 de novo pMHC
minibinder work (Venkatesh, Majumder et al.), where designs were kept when
the on-target peptide scored higher under MPNN than off-target peptides.

## 1. Setup

In [ ]:
!git clone --depth 1 https://github.com/dauparas/ProteinMPNN.git /content/mpnn_repo
!pip install -q gemmi numpy torch

## 2. Convert designed complexes to PDB and build the apo controls

BoltzGen writes CIF. ProteinMPNN reads PDB. Chain layout in the designed
complexes:

| chain | contents | length |
|---|---|---|
| A | de novo binder | 55-78 |
| B | HLA-A\*11:01 alpha1-alpha2 | 182 |
| C | beta2-microglobulin | 99 |
| D | KRAS G12D peptide VVVGADGVGK | 10 |

The apo control is the identical file with chain A removed - same backbone,
same peptide coordinates, same everything, minus the binder. Nothing is
relaxed or re-predicted between the two, so the only difference entering the
comparison is the binder's presence.

In [ ]:
import gemmi, glob, os
from pathlib import Path

SRC = "/content/designs/run/intermediate_designs_inverse_folded"
Path("/content/work/pdb_complex").mkdir(parents=True, exist_ok=True)
Path("/content/work/pdb_apo").mkdir(parents=True, exist_ok=True)

for f in sorted(glob.glob(f"{SRC}/pmhc_g12d_*.cif")):
    name = os.path.basename(f)[:-4]

    st = gemmi.read_structure(f); st.setup_entities()
    st.write_pdb(f"/content/work/pdb_complex/{name}.pdb")

    st = gemmi.read_structure(f); st.setup_entities()
    for model in st:
        for ch in [c.name for c in model]:
            if ch == "A":
                model.remove_chain(ch)
    st.write_pdb(f"/content/work/pdb_apo/{name}.pdb")

st = gemmi.read_structure("/content/work/pdb_complex/pmhc_g12d_06.pdb")
print("complex:", [(c.name, len(c)) for c in st[0]])

## 3. Conditional probabilities

`--pdb_path_chains D` marks the peptide as the designed chain; every other
chain is held fixed as sequence context. `--conditional_probs_only 1` skips
sequence generation and returns `p(s_i | backbone, s_rest)` directly, averaged
over 10 random decoding orders.

Vanilla weights, not the soluble model - this is a probability readout on a
fixed structure, not a design run.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/content/mpnn_repo")

def mpnn(pdb: Path, out: Path) -> Path:
    npz = out / "conditional_probs_only" / f"{pdb.stem}.npz"
    if npz.exists():
        return npz
    subprocess.run([sys.executable, "protein_mpnn_run.py",
                    "--pdb_path", str(pdb), "--pdb_path_chains", "D",
                    "--conditional_probs_only", "1", "--out_folder", str(out),
                    "--seed", "37", "--batch_size", "1",
                    "--num_seq_per_target", "10"],
                   cwd=REPO, check=True, capture_output=True)
    return npz

## 4. Parse and score

`log_p` has shape `(decoding_orders, L, 21)` over the alphabet
`ACDEFGHIKLMNPQRSTVWYX`. Average the probabilities across decoding orders,
renormalise, and read off the 10 positions flagged by `design_mask`.

`gly_control` averages P(Gly) at peptide positions 4, 7 and 9, which are
glycine in both the mutant and the wild type. It is a sanity check on the
structure rather than a result: if the model cannot recover those glycines
the backbone is poor and the p6 number carries no weight.

In [ ]:
import csv
import numpy as np

ALPH = "ACDEFGHIKLMNPQRSTVWYX"
WORK = Path("/content/work")
P6, PEP = 5, 10

def probs(npz: Path) -> np.ndarray:
    d = np.load(npz)
    idx = np.where(d["design_mask"] > 0)[0]
    assert len(idx) == PEP, f"{npz.name}: {len(idx)} designed positions"
    p = np.exp(d["log_p"]).mean(0)
    return (p / p.sum(-1, keepdims=True))[idx]

rows = []
for pdb in sorted(WORK.glob("pdb_complex/*.pdb")):
    pc = probs(mpnn(pdb, WORK / "out_complex"))
    pa = probs(mpnn(WORK / "pdb_apo" / pdb.name, WORK / "out_apo"))

    d_c, g_c = pc[P6, ALPH.index("D")], pc[P6, ALPH.index("G")]
    d_a, g_a = pa[P6, ALPH.index("D")], pa[P6, ALPH.index("G")]
    lo_c, lo_a = float(np.log(d_c / g_c)), float(np.log(d_a / g_a))

    rows.append({
        "design": pdb.stem,
        "pD6_complex": round(float(d_c), 4),
        "pG6_complex": round(float(g_c), 4),
        "logodds_complex": round(lo_c, 3),
        "pD6_apo": round(float(d_a), 4),
        "pG6_apo": round(float(g_a), 4),
        "logodds_apo": round(lo_a, 3),
        "delta_logodds": round(lo_c - lo_a, 3),
        "gly_control": round(float(np.mean(
            [pc[i, ALPH.index("G")] for i in (3, 6, 8)])), 3),
    })

rows.sort(key=lambda r: -r["delta_logodds"])
with open("mpnn_specificity.csv", "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

for r in rows[:8]:
    print(f"{r['design']}  delta {r['delta_logodds']:+.3f}  "
          f"gly_control {r['gly_control']:.2f}")

## 5. Result

The apo baseline sits at log-odds 2.81-3.13 across all 25 structures: the
groove alone prefers Asp over Gly at p6 by roughly a factor of 20, and it does
so consistently, which is a good sign that the measurement itself is stable.

Against that baseline no design reaches a meaningful positive delta. Two sit
at the zero line (+0.08, +0.02) and both have weak gly controls (0.59, 0.76),
so their backbones are the least reliably read of the set. Every design whose
gly control exceeds 0.85 - the structurally clean ones - returns a negative
delta between -0.35 and -1.40.

**None of these binders adds direct readout of the aspartate, and the better
formed ones slightly occlude the groove's own.**

That is the result the burial analysis predicted. Asp6 presents 8.8 square
angstroms of solvent-accessible surface, one carboxylate oxygen is fully
buried, and the therapeutic TCR in 7OW6 contacts none of it. A binder seated
above the groove has no physical route to the side chain, so a model asked
what the structure wants at p6 learns nothing extra from the binder's
presence.

Discrimination therefore has to come from somewhere other than direct
contact: from the backbone conformation of the p4-p7 bulge, or from its
dynamics. The free-state ensemble already measured a difference there - the
wild-type bulge spreads 0.95 A across ColabFold models against 0.44 A for
G12D, with anchor residues identical at 0.19 and 0.18 A. Molecular dynamics
on the complex is the direct test of whether a binder can sense that
difference in rigidity, and this result is what motivates running it.